# 10 - Log the Phase 2 benchmark evidence to W&B

The M8 benchmark table in the Phase 2 report comes from saved JSON files (`report/evidence/M8_benchmark_*.json`).
This notebook logs those files, unchanged, to the team's W&B project as one run with an artifact, so every number in the table can be traced to a W&B run.
It does **not** re-run any model and does not change any value.

Run time: about 2 minutes, CPU runtime. You will be asked to sign in to W&B (paste your API key into the Colab prompt, never into the notebook text).


In [ ]:
# 1. Code and evidence from GitHub
!git clone -q https://github.com/Yasser-20-alhajlah/smart-pet-vision /content/spv-repo 2>/dev/null || git -C /content/spv-repo pull -q
!pip -q install wandb==0.28.1
import json, csv, hashlib, pathlib
EV = pathlib.Path("/content/spv-repo/smart-pet-vision/phase2/report/evidence")
files = sorted(p for p in EV.glob("M8_*") if p.suffix in {".json", ".csv"})
for p in files:
    print(f"{p.name:45s} sha256={hashlib.sha256(p.read_bytes()).hexdigest()[:16]}")


In [ ]:
# 2. Sign in and log one run with the saved values and the files as an artifact
import wandb
wandb.login()   # Colab shows a prompt for the API key
run = wandb.init(entity="yasser-alhajlah-kaust", project="smart-pet-vision",
                 name="deploy-benchmark-evidence", group="phase2-evidence-2026-10-31",
                 job_type="evidence", tags=["phase2", "M8", "evidence-only"],
                 notes="Logs the saved M8 benchmark, service, parity and acceptance files used in the Phase 2 report. No model was re-run.")
pt = json.loads((EV / "M8_benchmark_pytorch.json").read_text())
ox = json.loads((EV / "M8_benchmark_onnx_detector.json").read_text())
svc = json.loads((EV / "M8_service_http.json").read_text())
for prefix, b in (("pytorch", pt), ("onnx_detector", ox)):
    for k, v in b["latency_ms"].items():
        run.summary[f"{prefix}/latency_{k}_ms"] = v
    run.summary[f"{prefix}/throughput_img_per_s"] = b["throughput_img_per_s"]
    run.summary[f"{prefix}/peak_gpu_mem_mb"] = b["peak_gpu_mem_mb"]
    run.summary[f"{prefix}/peak_process_rss_mb"] = b["peak_process_rss_mb"]
    run.summary[f"{prefix}/conditions"] = f'{b.get("device")}, batch {b.get("batch_size")}, warm-up {b.get("warmup")}, timed {b.get("runs")}'
for k in ("success_rate", "n", "requests"):
    if k in svc: run.summary[f"service/{k}"] = svc[k]
rows = list(csv.DictReader(open(EV / "M8_acceptance.csv")))
run.log({"acceptance": wandb.Table(columns=list(rows[0].keys()), data=[list(r.values()) for r in rows])})
art = wandb.Artifact("phase2-m8-evidence", type="evidence",
                     description="Saved M8 evidence files used in the Phase 2 report (unchanged)")
for p in files:
    art.add_file(str(p))
run.log_artifact(art)
url = run.url
run.finish()
print("W&B run:", url)


In [ ]:
# 3. Save the run link to Drive so it can be added to the report
from google.colab import drive
drive.mount("/content/drive")
open("/content/drive/MyDrive/wandb_evidence_run.txt", "w").write(url + "\n")
print("Saved /content/drive/MyDrive/wandb_evidence_run.txt:", url)
